# Derivative Trades (D-IFI)

Generates the D-IFI report for derivative trades using the TypeScript/Deno library.

In [ ]:
// ── EDIT THESE VALUES ───────────────────────────────────────────────
import { DateTime } from "luxon";
import {
  createContainer,
  NodeInfoProvider,
  IbkrBrokerageExportProvider,
  StagingFinancialGroupingProcessor,
  ApplyIdentifierRelationshipsService,
  KdvpReportGenerator,
  DivReportGenerator,
  IfiReportGenerator,
  SlovenianTaxAuthorityProvider,
  SlovenianTaxAuthorityReportTypes,
  TaxAuthorityLotMatchingMethod,
  TaxPayerType,
} from "@brrr/lib";
import type { TaxAuthorityConfiguration, TaxPayerInfo } from "@brrr/lib";

const importsDir = "./imports";    // folder containing IBKR XML exports
const exportsDir = "./exports";    // folder for generated output files

const taxPayerInfo: TaxPayerInfo = {
  taxNumber: "12345678",
  taxpayerType: TaxPayerType.PHYSICAL_SUBJECT,
  name: "Ime Priimek",
  address1: "Ulica 1",
  address2: "",
  city: "Ljubljana",
  postNumber: "1000",
  postName: "Ljubljana",
  municipalityName: "Ljubljana",
  birthDate: DateTime.fromISO("1990-01-01"),
  maticnaStevilka: "",
  invalidskoPodjetje: false,
  resident: true,
  activityCode: "",
  activityName: "",
  countryId: "SI",
  countryName: "Slovenia",
};

const reportConfig: TaxAuthorityConfiguration = {
  fromDate: DateTime.fromISO("2025-01-01"),
  toDate: DateTime.fromISO("2026-01-01"),
  lotMatchingMethod: TaxAuthorityLotMatchingMethod.FIFO,
};

const container = createContainer(new NodeInfoProvider());

const provider = new SlovenianTaxAuthorityProvider(
  taxPayerInfo,
  reportConfig,
  container.get(ApplyIdentifierRelationshipsService),
  container.get(KdvpReportGenerator),
  container.get(DivReportGenerator),
  container.get(IfiReportGenerator),
);

## Load & Process Broker Exports

In [ ]:
const ibkrProvider = container.get(IbkrBrokerageExportProvider);
const groupingProcessor = container.get(StagingFinancialGroupingProcessor);

const xmlContents: string[] = [];
for await (const entry of Deno.readDir(importsDir)) {
  if (entry.isFile && entry.name.endsWith(".xml")) {
    xmlContents.push(await Deno.readTextFile(`${importsDir}/${entry.name}`));
  }
}
console.log(`Found ${xmlContents.length} XML file(s)`);

const stagingEvents = ibkrProvider.loadAndTransform(xmlContents);
const financialEvents = groupingProcessor.processStagingFinancialEvents(stagingEvents);

console.log(`Processed ${financialEvents.groupings.length} grouping(s)`);

## Identifier Relationships

In [ ]:
import { IdentifierChangeType } from "@brrr/lib";

const applyService = container.get(ApplyIdentifierRelationshipsService);
// Note: SlovenianTaxAuthorityProvider also applies relationships internally.
// This cell is just for inspection.
const inspectedEvents = applyService.apply(financialEvents, [
  IdentifierChangeType.RENAME,
  IdentifierChangeType.SPLIT,
  IdentifierChangeType.REVERSE_SPLIT,
]);

if (inspectedEvents.identifierRelationships.length === 0) {
  console.log("No identifier relationships (no renames/splits found).");
} else {
  for (const rel of inspectedEvents.identifierRelationships) {
    const from = rel.fromIdentifier.getIsin() ?? rel.fromIdentifier.getTicker() ?? "?";
    const to   = rel.toIdentifier.getIsin()   ?? rel.toIdentifier.getTicker()   ?? "?";
    console.log(`${rel.changeType}: ${from} → ${to} (${rel.effectiveDate.toISODate()})`);
  }
}

## Generate D-IFI Report (Derivative Trades)

In [ ]:
const reportType = SlovenianTaxAuthorityReportTypes.D_IFI;

const xmlOutput = await provider.generateExportForTaxAuthority(reportType, financialEvents);
const csvOutput = await provider.generateSpreadsheetExport(reportType, financialEvents);

await Deno.mkdir(exportsDir, { recursive: true });
await Deno.writeTextFile(`${exportsDir}/D_Ifi.xml`, xmlOutput);
await Deno.writeTextFile(`${exportsDir}/export-derivatives.csv`, csvOutput);

console.log(`Written: ${exportsDir}/D_Ifi.xml`);
console.log(`Written: ${exportsDir}/export-derivatives.csv`);